# **06. Cluster Experiment --- Ames Housing Dataset**

## **1. Dataset Info**

In [37]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable
import sklearn
import feature_engine
import scipy
import kaggle
import os
import zipfile
from pathlib import Path
import pickle
import joblib
%matplotlib inline
sns.set_style('darkgrid')


In [38]:
# dispaly setting
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [39]:
# Define path to read files
parent_path = Path.cwd().parent
data_path = parent_path.joinpath("data", "raw")
processed_path = parent_path.joinpath("data", "processed")

# will read original dataset
files = []
for file in data_path.rglob("*.csv"):
    files.append(file)
    print(files.index(file), " ", file.name)

0   sample_submission.csv
1   test.csv
2   train.csv


In [40]:
train_org = pd.read_csv(files[2])
test_org = pd.read_csv(files[1])

In [41]:
# Will read the processed data which stores in notebook "01-Data-Cleaning"
clean_files = []
for file in processed_path.glob("*"):
    clean_files.append(file)
    print(clean_files.index(file), " ", file.name)

0   .gitkeep
1   test_cleaned.csv
2   train_cleaned.csv


In [42]:
print("Loading Cleaned data....")
train_cleaned = pd.read_csv(clean_files[1])
test_cleaned = pd.read_csv(clean_files[2])


Loading Cleaned data....


In [43]:
# Data type in train and test datasets should be same for all columns. Let's check that.
print(train_cleaned.dtypes.value_counts())
print(test_cleaned.dtypes.value_counts())

object     43
int64      26
float64    11
Name: count, dtype: int64
object     43
int64      35
float64     3
Name: count, dtype: int64


In [44]:
train_cleaned['GarageYrBlt'] = train_cleaned['GarageYrBlt'].astype('int64')
test_cleaned[['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt',
       'GarageCars', 'GarageArea']] = test_cleaned[['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageYrBlt',
       'GarageCars', 'GarageArea']].astype('int64')

In [45]:
# check data types of train and test datasets after conversion
print(train_cleaned.dtypes.value_counts())
print(test_cleaned.dtypes.value_counts())

object     43
int64      27
float64    10
Name: count, dtype: int64
object     43
int64      36
float64     2
Name: count, dtype: int64


**Segregrate the features**

In [47]:
numeric_cols = train_org.select_dtypes(include='number').columns
cat_cols = train_org.select_dtypes(include='object').columns

In [48]:
train_org.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [50]:
# selecting rows where yrsold > yearremodeled
train_df = train_org[train_org['YrSold'] >= train_org['YearRemodAdd']]
test_df = test_org[test_org['YrSold'] >= test_org['YearRemodAdd']]

In [51]:
print(train_df.shape)
print(test_df.shape)

(1459, 81)
(1457, 80)


In [52]:
print(test_org.shape)
print(test_org.shape)

(1459, 80)
(1459, 80)


## **2. Missing Value Analysis**

In [53]:
train_df.stb.missing().query("percent > 0")
# Crietria : Drop features with above 6% missing values.

,missing,total,percent
PoolQC,1452,1459,99.520219
MiscFeature,1405,1459,96.298835
Alley,1368,1459,93.762851
Fence,1178,1459,80.740233
MasVnrType,872,1459,59.766964
FireplaceQu,690,1459,47.292666
LotFrontage,259,1459,17.751885
GarageYrBlt,81,1459,5.551748
GarageCond,81,1459,5.551748
GarageType,81,1459,5.551748
